# Rotary Position Embedding (RoPE)
*Encoding token positions by rotating embedding vectors — the positional scheme in LLaMA, Mistral, Qwen, GLM.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_pair_coalesced.py`


## What Is RoPE?

**In plain English:** RoPE encodes the *position* of each token by rotating its query/key vectors. A rotation doesn't change the vector's length, only its direction. The key insight: when you dot-product two rotated vectors, the result depends only on their *relative* positions — not their absolute positions. This means attention naturally learns position-relative patterns (e.g., "adjacent" vs "far away").

**Why not learned positional embeddings?** Learned embeddings can't generalize beyond the training context length. RoPE extrapolates to longer sequences because it encodes relative offsets algebraically.


In [ ]:
import math
print('ready')

## Worked Example: 2D Rotation ($d = 2$)

A 2D vector $[x_0, x_1]$ rotated by angle $\theta$ becomes:

$$\begin{bmatrix} x_0' \\ x_1' \end{bmatrix} = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}$$

Take $x = [1.0, 0.0]$ (a unit vector pointing right).

**Position $m=0$, rotation angle $= 0 \cdot 1.0 = 0$ rad:**

$$x' = [\cos(0), \sin(0)] = [1.0,\ 0.0] \quad \text{(unchanged)}$$

**Position $m=1$, rotation angle $= 1 \cdot 1.0 = 1.0$ rad ($\approx 57°$):**

$$x' = [\cos(1.0),\ \sin(1.0)] = [0.540,\ 0.841]$$

### The Relative Position Property

$\text{RoPE}(x, m=0) \cdot \text{RoPE}(x, m=1) = [1.0, 0.0] \cdot [0.540, 0.841] = 0.540$

This dot product depends only on the **difference** $\Delta m = 1$, not on the absolute positions.

### The Norm is Preserved (Isometry)

$\|[0.540, 0.841]\| = \sqrt{0.540^2 + 0.841^2} = \sqrt{0.292 + 0.707} = 1.0$ ✅

Rotation can never change the vector's length — only its direction.


In [ ]:
def rope_2d(x, m, theta=1.0):
    angle = m * theta
    c, s = math.cos(angle), math.sin(angle)
    return [c * x[0] - s * x[1],
            s * x[0] + c * x[1]]

x = [1.0, 0.0]
for m in range(4):
    xr = rope_2d(x, m)
    norm = math.sqrt(xr[0]**2 + xr[1]**2)
    angle_deg = m * 1.0 * 180 / math.pi
    print(f"  m={m}:  angle={angle_deg:6.1f}°  "
          f"x' = [{xr[0]:+.4f}, {xr[1]:+.4f}]  "
          f"|x'| = {norm:.6f}")

print()
# Relative position property: dot(m=0, m=1) vs dot(m=2, m=3) should be equal
xr0 = rope_2d(x, 0);  xr1 = rope_2d(x, 1)
xr2 = rope_2d(x, 2);  xr3 = rope_2d(x, 3)
dot_01 = xr0[0]*xr1[0] + xr0[1]*xr1[1]
dot_23 = xr2[0]*xr3[0] + xr2[1]*xr3[1]
print(f"dot(pos=0, pos=1) = {dot_01:.6f}")
print(f"dot(pos=2, pos=3) = {dot_23:.6f}")
print(f"Same? {abs(dot_01 - dot_23) < 1e-9}  ← relative distance Δm=1 → same dot product")


## The Formula (General $d$-Dimensional Case)

Split the $d$-dimensional vector into $d/2$ pairs. For pair $k$ (dimensions $2k$ and $2k+1$):

$$\theta_k = \frac{1}{10000^{2k / d}}$$

$$\begin{bmatrix} x_{2k}' \\ x_{2k+1}' \end{bmatrix} = \begin{bmatrix} \cos(m \theta_k) & -\sin(m \theta_k) \\ \sin(m \theta_k) & \cos(m \theta_k) \end{bmatrix} \begin{bmatrix} x_{2k} \\ x_{2k+1} \end{bmatrix}$$

| Symbol | Meaning |
|--------|---------|
| $m$ | Token position (0, 1, 2, …) |
| $k$ | Dimension pair index (0, 1, …, $d/2 - 1$) |
| $\theta_k$ | Base frequency for pair $k$ |
| $10000$ | The "base" (hyperparameter, sometimes 500000 for long context) |
| $\cos(m\theta_k), \sin(m\theta_k)$ | Rotation coefficients for position $m$ |

### 🗣️ Say It Out Loud
> *"For each dimension pair k, we rotate the pair by angle m times theta-sub-k. Theta-sub-k equals 1 over 10000 to the power of 2k divided by d."*

**Tutor's gloss:** "Different pairs rotate at different speeds. Pair 0 has $\theta_0 = 1.0$ — it spins fast (full rotation in ~6 positions). Pair $d/2-1$ has $\theta \approx 0$ — it spins very slowly (full rotation in ~600k positions). Fast pairs encode local (adjacent) relationships; slow pairs encode long-range (paragraph-level) context."


In [ ]:
def rope_frequencies(d, base=10000):
    return [1.0 / (base ** (2*k / d)) for k in range(d // 2)]

def rope(x, m, d, base=10000):
    thetas = rope_frequencies(d, base)
    result = list(x)
    for k, theta in enumerate(thetas):
        angle = m * theta
        c, s = math.cos(angle), math.sin(angle)
        i, j = 2*k, 2*k+1
        result[i] = c * x[i] - s * x[j]
        result[j] = s * x[i] + c * x[j]
    return result

d = 8
thetas = rope_frequencies(d)
print(f"Frequencies θ_k for d={d}:")
for k, th in enumerate(thetas):
    period = 2 * math.pi / th if th > 0 else float('inf')
    print(f"  k={k}: θ={th:.5f}  full rotation every {period:.1f} tokens")

print()
x_test = [1.0] * d
for m in [0, 1, 100, 1000]:
    xr = rope(x_test, m, d)
    norm = math.sqrt(sum(v**2 for v in xr))
    print(f"  m={m:5d}:  norm={norm:.6f}  (always 1 — isometry ✓)")


## Why RoPE Works Better Than Learned Positional Embeddings

| Property | Learned PE | RoPE |
|----------|-----------|------|
| Generalizes beyond training length | ❌ | ✅ (extrapolates by design) |
| Encodes relative positions | ❌ (absolute only) | ✅ (by the rotation algebra) |
| Parameter count | $T_\text{max} \times d$ | 0 (no parameters) |
| Context length extension | Requires retraining | Adjust base ($10000 \to 500000$) |

Used by: LLaMA 2/3, Mistral, Qwen, GLM-4/5, Gemma, Phi, Falcon.


## Optimization Ladder

| Version | Strategy | Key constraint |
|---------|----------|---------------|
| `v0_naive` | One thread per element, scalar loads | Under-utilizes memory bandwidth |
| `sm89_v1_pair_coalesced` | One thread per (head, pair), vectorized pair load | Near-optimal memory throughput |

**Why RoPE is simple to optimize:**
Each thread handles one (position, head, pair) independently — no reduction, no shared memory, no synchronization needed. Threads never talk to each other. This means:
- No tree reduce rounds
- No shared memory allocation
- No sync_threads() calls

The only optimization is making sure adjacent threads read adjacent memory locations (coalesced access), which the `v1` kernel achieves by assigning thread $k$ to dimension pair $k$.

## RTX 4080: RoPE is Memory-Bound by Design

For each dimension pair, RoPE does 6 FLOPs:
- 2 multiplies and 1 subtract for `x0' = x0·cos − x1·sin`
- 2 multiplies and 1 add for    `x1' = x0·sin + x1·cos`

It moves 12 bytes per pair: 2 BF16 reads, 2 BF16 writes, 2 freq-table reads (cos and sin).
Arithmetic intensity: 6 ÷ 12 = **0.5 FLOP/byte** — far below the RTX 4080's 151 F/B ridge.

**The frequency table is cached for free.**
The table shape is `[seq_len, d//2]`. For Qwen3-8B at seq_len=4096: 4096 × 32 × 4 bytes = 512 KB.
The RTX 4080 has 64 MB of L2 cache. The table fits easily.
Every attention head reads the same frequencies, so after the first head,
all subsequent heads get the table from L2 at ~7 TB/s — effectively zero cost.

**RoPE is rarely a bottleneck.**
At one decode step with Q shape [1, 32, 128]: only 8 KB to read and write total.
That's done in a handful of microseconds.

## CuTeDSL Kernel Walkthrough

```python
@cute.kernel
def rope_kernel(mX, mFreqCos, mFreqSin, mOut):
    # Launch config: one CTA per (sequence position × head)
    seq_pos = blockIdx.x    # which token in the sequence
    head    = blockIdx.y    # which attention head (Q or K)
    k       = threadIdx.x   # which dimension pair (0 to d//2 - 1)

    # Load the two adjacent dimensions that form this pair
    # In memory, dim 2k and dim 2k+1 are right next to each other
    x0 = mX[seq_pos, head, 2*k]       # even dimension
    x1 = mX[seq_pos, head, 2*k + 1]   # odd dimension

    # Load the precomputed rotation angles for this position and pair
    # These were computed once at model load time, not per-token
    cos_m = mFreqCos[seq_pos, k]   # cos(position × theta_k)
    sin_m = mFreqSin[seq_pos, k]   # sin(position × theta_k)

    # Apply the 2D rotation:
    # [x0']   [cos  -sin] [x0]
    # [x1'] = [sin   cos] [x1]
    mOut[seq_pos, head, 2*k]     = x0 * cos_m - x1 * sin_m
    mOut[seq_pos, head, 2*k + 1] = x0 * sin_m + x1 * cos_m
    # 6 FLOPs. Done. No communication with other threads.
```

**The key insight:** threads are completely independent.
Thread 0 handles pair 0, thread 1 handles pair 1, ... zero coordination.
For head_dim=128: 64 threads per head, all running simultaneously.
The SM89 can run 2048 threads per SM, so many heads and positions run in parallel with no waiting.

## Qwen3-8B Specifics: NoPE Dimensions and GPU Utilization

Qwen3-8B uses a split head design: `head_dim=128` but `rope_head_dim=64`. Only the **first 64 dimensions** of each head get rotated. The **last 64 (NoPE) pass through unchanged**.

```
Each head vector [128 dims]:
  dims 0..63   → apply RoPE (32 frequency pairs)
  dims 64..127 → copy unchanged ("No Positional Encoding")
```

This means the RoPE kernel needs to:
1. Load all 128 elements per head
2. Rotate the first 64
3. Pass through the last 64
4. Write all 128 back

**The NoPE copy is free** — it happens in the same kernel pass, no extra HBM reads.

**SM utilization analysis at decode (T=1, generating one token):**
```
Q after q_proj: shape [1, 32_heads, 128]  → 32 heads to process
K after k_proj: shape [1,  8_heads, 128]  → 8 heads to process
Total: 40 heads × 64 pairs = 2,560 threads

Launch config: 64 threads per block (one per pair) → 40 blocks
RTX 4080 can run 76 SMs × (1024/64) = 76 × 16 = 1,216 blocks simultaneously
We're launching 40 → GPU is 40/1,216 = 3.3% utilized.
```

RoPE at decode is **negligible** — it uses 3% of the GPU for a few microseconds. It will never show up in a profiler at decode time.

**At prefill (T=2048):**
```
Q shape: [2048, 32, 128] → 2048 × 32 = 65,536 head-position pairs
Launch: 65,536 blocks of 64 threads = ~4M threads
Easily saturates all 76 SMs → RoPE takes ≈1 ms but is properly parallelized.
```

**Practical suggestion — fuse RoPE into the Q/K projection epilogue at decode:**

At decode, launching a separate RoPE kernel for 40 blocks is wasteful — most of the "time" is kernel launch overhead, not compute. Fusing RoPE into the GEMM epilogue eliminates this:

```python
# In the GEMM epilogue, after computing the output register tile rC:
for thread_elem in range(elems_per_thread):
    out_dim = thread_elem_to_dim(thread_elem)
    if out_dim < 64:                        # RoPE region
        pair = out_dim // 2
        cos_m = freq_cos[seq_pos, pair]
        sin_m = freq_sin[seq_pos, pair]
        x0 = rC[2*pair];  x1 = rC[2*pair+1]
        rC[2*pair]   = x0 * cos_m - x1 * sin_m
        rC[2*pair+1] = x0 * sin_m + x1 * cos_m
    # else: NoPE region — rC unchanged
mOut[...] = rC   # write once to HBM
```

This saves one HBM read + write of the [1, 40, 128] tensor (= 10 KB — negligible for correctness but eliminates a kernel launch overhead that can be 5–20 µs on its own).

## Where RoPE Lives in the Qwen3-8B Decode Step

RoPE is applied after the Q and K projections, before the attention dot-product.
It runs at step ⑦ and ⑧ of each transformer layer (using the numbering from the full decode flow in the RMSNorm notebook):

```
③ Q projection  W[4096,4096]  → q [1, 4096]  (reshape → [1, 32_heads, 128])
④ K projection  W[1024,4096]  → k [1, 1024]  (reshape → [1,  8_heads, 128])
⑤ q_norm (RMSNorm per head)   → q normalized
⑥ k_norm (RMSNorm per head)   → k normalized
⑦ RoPE on Q:  rotate dims 0..63 of each of 32 heads  → q_rope [1, 32, 128]
⑧ RoPE on K:  rotate dims 0..63 of each of  8 heads  → k_rope [1,  8, 128]
⑨ Append k_rope, v to KV cache
⑩ GQA attention: q_rope × k_cache.T → scores → softmax → × v_cache
```

### What RoPE rotates and what it doesn't

Each head vector has 128 dimensions. Only the first 64 are rotated:

```
head vector [128 dims]:
  dims  0..63  → 32 frequency pairs, each rotated by angle = position × theta_k
  dims 64..127 → NoPE: passed through unchanged (no positional encoding applied)
```

### Why K gets RoPE baked in at cache-write time

At decode step $t$, Q is at position $t$. Cached keys are at positions $0, 1, ..., t-1$.
RoPE is applied to K when it is first computed and stored in the cache — the rotation angle is baked into the stored vector. When attention computes `Q[t] · K[s]`, the relative rotation `theta_k × (t - s)` appears naturally in the dot product. This is the "relative position" property.

### Decode vs prefill RoPE cost

```
Decode:  Q shape [1, 32, 128] → 32 × 64 = 2,048 elements to rotate → microseconds
Prefill: Q shape [2048, 32, 128] → 2048 × 32 × 64 = 4,194,304 elements → 2048× more work
```

At decode, RoPE is negligible. At prefill it scales with the sequence length but remains fast (no reduction, trivially parallel). It is never a bottleneck in either regime.

## RoPE Implementations in the Wild

### HuggingFace Transformers

`modeling_qwen3.py::Qwen3RotaryEmbedding` — computes the freq table at init, applies via `apply_rotary_pos_emb`.
Pure PyTorch: `torch.cos` / `torch.sin` + view-as-complex multiplication.
Supports `rope_scaling` dict for YaRN, linear, dynamic, and LongRoPE variants.

### FlashInfer

`flashinfer.rope.apply_rope_inplace(q, k, indptr, offsets, rope_scale, rope_theta)`
- CUDA C++ kernel, in-place, handles variable-length sequences (required for paged attention)
- `flashinfer.rope.apply_rope_with_cos_sin_cache` skips recomputing trig each call by reading from a precomputed table
- For Q: called once per decode step with shape `[1, 32, 128]`
- For K: called at prompt ingestion and at each decode step for the new key before it enters the KV cache

### vLLM

`vllm/model_executor/layers/rotary_embedding.py` — dispatches to a Triton or CUDA kernel.
In paged attention mode, RoPE is fused into the `PagedAttention` kernel: keys are rotated at the moment they are written into the paged KV cache, not in a separate pass.

### TRT-LLM

RoPE is fused into the `FusedMHA` TensorRT plugin and never visible as a standalone kernel in the execution plan. The TRT compiler determines optimal scheduling automatically.

### Pattern across all implementations

1. **Precompute the freq table once** at model load time — not per-token
2. **In-place or fused** — avoid a separate HBM write/read cycle for Q and K
3. **NoPE pass-through** — only rotate `rope_head_dim` dims; copy the remaining dims unchanged
4. **Conjugate for output** — some Qwen3 implementations undo the rotation on the attention output vector

### The key optimization gap in this project

The CuTeDSL `sm89_v1_pair_coalesced` kernel is the next step.
The performance win is small (RoPE is never the bottleneck), but the implementation
teaches vectorized pair loads and the thread-independent kernel structure that appears throughout CuTeDSL.

## One-Sentence Takeaway

RoPE is the lightest-weight operation in the entire transformer: each thread independently rotates one dimension pair with 6 FLOPs and zero synchronization, the frequency table is computed once at model load and cached permanently in L2, and at decode the entire kernel finishes in microseconds — making it a perfect first CuTeDSL exercise but never an inference bottleneck, with the real optimization being fusion into the surrounding Q/K projection GEMMs to eliminate a kernel launch rather than improving the rotation arithmetic itself.

## Community Optimization Resources for RoPE on SM89

### Speedup reference (community benchmarks on SM89-adjacent hardware)

| Optimization | Hardware | Before | After | Speedup | Implementation | What was fused |
|---|---|---|---|---|---|---|
| M-RoPE (Multimodal) eager→Triton | RTX 5090 | HF eager | Triton fused | **9.87×** | Triton | Multi-modal RoPE computation |
| QKNorm + RoPE fused | MI300X → SM89 analog | 2 separate kernels | 1 kernel | **2.27×** | Triton | QK-norm + rotation combined |
| RoPE in-place vs copy | RTX 4090 (SM89) | allocate new tensor | in-place rotate | ~1.5× | CUDA / Triton | Eliminates output allocation |
| RoPE fused into QKV projection | RTX 4090 (SM89) | separate kernel | GEMM epilogue | ~2× | CUTLASS / CuTeDSL | Eliminates HBM round-trip for Q, K |

**Source:** unsloth/unsloth M-RoPE benchmark; FlashInfer rope benchmarks

---

### Why RoPE is never the bottleneck — but fusion saves launches

At decode (generating one token):
- Q shape: `[1, 32, 128]` → 32 × 64 pairs = 2,048 elements → ~8 KB
- Kernel launch overhead: 5–20 µs
- RoPE compute time: <1 µs

**The real cost is the kernel launch, not the FLOPs.** The 9.87× M-RoPE speedup vs HF eager comes almost entirely from eliminating Python overhead and kernel-launch round-trips, not from improving the rotation math.

---

### Qwen3-8B specifics: `rope_head_dim=64` and NoPE

```
Each head: [128 dims total]
  dims 0..63   → RoPE: 32 frequency pairs, theta_k = 1 / (1_000_000 ^ (2k/64))
  dims 64..127 → NoPE: copied unchanged
  
rope_theta = 1,000,000  (10× higher than standard LLaMA-2 base)
→ Low-freq pairs rotate 10× slower → better long-range generalization
```

The frequency table for Qwen3 is precomputed at model load:
```
seq_len=4096, 32 pairs, 2 values (cos+sin), 4 bytes = 1 MB → fits in L2 permanently
seq_len=32768 → 8 MB → still fits in 64 MB L2
```

---

### CuTeDSL: fuse RoPE into the QKV projection epilogue

Avoids a separate kernel launch and one HBM write+read of Q and K:

```python
# In the GEMM epilogue, after tensor cores produce rC:
for thread_elem in cute.range(elems_per_thread):
    out_dim = thread_elem_to_dim(thread_elem)
    if out_dim < 64:                            # RoPE region
        pair = out_dim // 2
        cos_m = freq_cos[seq_pos, pair]
        sin_m = freq_sin[seq_pos, pair]
        x0 = rC[2*pair]; x1 = rC[2*pair+1]
        rC[2*pair]   = x0 * cos_m - x1 * sin_m
        rC[2*pair+1] = x0 * sin_m + x1 * cos_m
    # dims 64..127: NoPE — rC unchanged
mOut[...] = rC   # one write to HBM, no intermediate RoPE tensor
```

**Saves at decode:** 1 HBM write + 1 HBM read of `[1, 32, 128]` = 16 KB → negligible bytes but eliminates a kernel launch (~10 µs).

---

### FlashInfer rope API (reference for what to replicate)

```python
# In-place, handles paged KV and variable-length sequences:
flashinfer.rope.apply_rope_inplace(q, k, indptr, offsets,
                                   rope_scale=1.0, rope_theta=1_000_000)

# With precomputed cos/sin cache (avoids recomputing trig each call):
flashinfer.rope.apply_rope_with_cos_sin_cache(q, k, cos_cache, sin_cache, ...)
```

The precomputed cache version is the right target for a CuTeDSL kernel — compute the table once at model load, then just load from it per token.